# Atom Training on Google Colab

Bootstrap, quick test, full run, and resume in one notebook.


## 1) Mount Drive


In [24]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Configure variables

Defaults are set for your current public repo/branch, but values are only set if missing.


In [25]:
import os

# Repo defaults (only applied when missing)
os.environ.setdefault('ATOM_REPO_URL', 'https://github.com/MBifolco/atom.git')
os.environ.setdefault('ATOM_BRANCH', 'colab')

# Runtime/cache defaults
os.environ.setdefault('ATOM_DRIVE_REPO', '/content/drive/MyDrive/dev/atom')
os.environ.setdefault('ATOM_WORK_REPO', '/content/atom')
os.environ.setdefault('ATOM_INSTALL_JAX_CUDA', '1')
os.environ.setdefault('ATOM_JAX_VERSION', '0.7.2')
os.environ.setdefault('ATOM_DRIVE_REPO_SYNC_MODE', 'stash')  # stash|reset|skip_pull

# Auth defaults (public repo workflow)
os.environ.setdefault('ATOM_USE_GITHUB_TOKEN', '0')
if os.environ.get('ATOM_USE_GITHUB_TOKEN') != '1':
    os.environ.pop('GITHUB_TOKEN', None)

# Private repo option:
# import getpass
# os.environ['ATOM_USE_GITHUB_TOKEN'] = '1'
# os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT: ').strip()

for key in [
    'ATOM_REPO_URL',
    'ATOM_BRANCH',
    'ATOM_DRIVE_REPO',
    'ATOM_WORK_REPO',
    'ATOM_INSTALL_JAX_CUDA',
    'ATOM_JAX_VERSION',
    'ATOM_DRIVE_REPO_SYNC_MODE',
    'ATOM_USE_GITHUB_TOKEN',
]:
    print(f"{key}={os.environ[key]}")
print('GITHUB_TOKEN set =', 'GITHUB_TOKEN' in os.environ)


ATOM_REPO_URL=https://github.com/MBifolco/atom.git
ATOM_BRANCH=colab
ATOM_DRIVE_REPO=/content/drive/MyDrive/dev/atom
ATOM_WORK_REPO=/content/atom
ATOM_INSTALL_JAX_CUDA=1
ATOM_JAX_VERSION=0.7.2
ATOM_DRIVE_REPO_SYNC_MODE=stash
ATOM_USE_GITHUB_TOKEN=0
GITHUB_TOKEN set = False


## 3) Clone (if needed) and bootstrap

This handles stale partial clones and branch checks before running `colab_bootstrap.sh`.


In [26]:
%%bash
set -euo pipefail

WORK_REPO="${ATOM_WORK_REPO:-/content/atom}"
REPO_URL="${ATOM_REPO_URL:-}"
BRANCH="${ATOM_BRANCH:-main}"
AUTH_REPO_URL="$REPO_URL"

echo "Using WORK_REPO=$WORK_REPO"
echo "Using BRANCH=$BRANCH"

if [[ -z "$REPO_URL" ]]; then
  echo "ERROR: ATOM_REPO_URL must be set before first clone."
  exit 1
fi

if [[ "$REPO_URL" == *"<org>"* || "$REPO_URL" == *"<repo>"* ]]; then
  echo "ERROR: ATOM_REPO_URL still contains placeholders. Set a real repo URL first."
  exit 1
fi

if [[ "${ATOM_USE_GITHUB_TOKEN:-0}" == "1" && -n "${GITHUB_TOKEN:-}" && "$REPO_URL" == https://github.com/* ]]; then
  AUTH_REPO_URL="${REPO_URL/https:\/\//https:\/\/${GITHUB_TOKEN}@}"
fi

if [[ -d "$WORK_REPO" && ! -d "$WORK_REPO/.git" ]]; then
  echo "Found stale non-git directory at $WORK_REPO. Removing it..."
  rm -rf "$WORK_REPO"
fi

if [[ ! -d "$WORK_REPO/.git" ]]; then
  echo "Cloning repository..."
  if ! git clone --branch "$BRANCH" --single-branch "$AUTH_REPO_URL" "$WORK_REPO"; then
    echo "ERROR: git clone failed."
    echo "Common causes:"
    echo "  1) Wrong ATOM_REPO_URL"
    echo "  2) Branch does not exist remotely"
    echo "  3) Private repo without token"
    echo "  4) Temporary GitHub/network issue"
    exit 1
  fi
fi

cd "$WORK_REPO"
chmod +x colab_bootstrap.sh
bash colab_bootstrap.sh


Using WORK_REPO=/content/atom
Using BRANCH=colab
Updating Drive repo cache (colab)...
M	archived/benchmarks/benchmark_gpu.py
M	archived/benchmarks/benchmark_jax_physics.py
M	archived/benchmarks/demo_jax_scaling.py
M	archived/legacy_training/training/train_fighter.py
M	archived/legacy_training/training/train_population.py
M	archived/setup_scripts/upgrade_rocm_to_7.sh
M	archived/tests/test_level3_integration.py
M	atom_fight.py
M	colab_bootstrap.sh
M	create_html_montage.py
M	create_montage.py
M	diagnose_level5_nan.py
M	render_replays.py
M	resume_population_training.py
M	setup_gpu.sh
M	tail_latest_log.sh
M	test_vmap_hang.py
Your branch is behind 'origin/colab' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating 5bb0a0f..255d96a


From https://github.com/MBifolco/atom
 * branch            colab      -> FETCH_HEAD
Already on 'colab'
From https://github.com/MBifolco/atom
 * branch            colab      -> FETCH_HEAD
error: Your local changes to the following files would be overwritten by merge:
	colab_bootstrap.sh
Please commit your changes or stash them before you merge.
Aborting


CalledProcessError: Command 'b'set -euo pipefail\n\nWORK_REPO="${ATOM_WORK_REPO:-/content/atom}"\nREPO_URL="${ATOM_REPO_URL:-}"\nBRANCH="${ATOM_BRANCH:-main}"\nAUTH_REPO_URL="$REPO_URL"\n\necho "Using WORK_REPO=$WORK_REPO"\necho "Using BRANCH=$BRANCH"\n\nif [[ -z "$REPO_URL" ]]; then\n  echo "ERROR: ATOM_REPO_URL must be set before first clone."\n  exit 1\nfi\n\nif [[ "$REPO_URL" == *"<org>"* || "$REPO_URL" == *"<repo>"* ]]; then\n  echo "ERROR: ATOM_REPO_URL still contains placeholders. Set a real repo URL first."\n  exit 1\nfi\n\nif [[ "${ATOM_USE_GITHUB_TOKEN:-0}" == "1" && -n "${GITHUB_TOKEN:-}" && "$REPO_URL" == https://github.com/* ]]; then\n  AUTH_REPO_URL="${REPO_URL/https:\\/\\//https:\\/\\/${GITHUB_TOKEN}@}"\nfi\n\nif [[ -d "$WORK_REPO" && ! -d "$WORK_REPO/.git" ]]; then\n  echo "Found stale non-git directory at $WORK_REPO. Removing it..."\n  rm -rf "$WORK_REPO"\nfi\n\nif [[ ! -d "$WORK_REPO/.git" ]]; then\n  echo "Cloning repository..."\n  if ! git clone --branch "$BRANCH" -

## 4) Sanity checks


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python - <<'PY'
import torch
from src.training.utils.runtime_platform import detect_runtime_platform
print('torch.cuda.is_available =', torch.cuda.is_available())
print('detected runtime platform =', detect_runtime_platform())
PY

nvidia-smi || true


## 5) Quick smoke test


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python train_progressive.py \
  --mode quick \
  --device cuda \
  --use-vmap \
  --output-dir /content/drive/MyDrive/atom_runs/quick_test


## 6) Full training run


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python train_progressive.py \
  --mode complete \
  --device cuda \
  --use-vmap \
  --timesteps 2000000 \
  --population 8 \
  --generations 10 \
  --output-dir /content/drive/MyDrive/atom_runs/run1


## 7) Resume interrupted run


In [ ]:
%%bash
set -euo pipefail
cd "${ATOM_WORK_REPO:-/content/atom}"

python resume_population_training.py \
  --checkpoint-dir /content/drive/MyDrive/atom_runs/run1 \
  --start-gen 8 \
  --total-gens 20 \
  --use-vmap


## Troubleshooting

- Public repo: keep `ATOM_USE_GITHUB_TOKEN=0`.
- Private repo: set `ATOM_USE_GITHUB_TOKEN=1` and provide `GITHUB_TOKEN`.
- If Drive cache has local edits, bootstrap now auto-stashes by default (`ATOM_DRIVE_REPO_SYNC_MODE=stash`).
- Set `ATOM_DRIVE_REPO_SYNC_MODE=reset` to discard cache edits, or `skip_pull` to avoid pulling when dirty.
